In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [8]:
# Load the CSV into a DataFrame
df = pd.read_csv('meteorite_landings.csv')

# Inspect the first few rows
df.head()
df.info()
# 1.4 Convert Mass → numeric (grams → kilograms)
df["mass_g"] = pd.to_numeric(df["Mass"], errors="coerce")    # grams
df["mass_kg"] = df["mass_g"] / 1000

# 1.5 Parse Coordinates → reclat, reclong
#    Assumes format "lat, long"
coords = df["Coordinates"].str.split(",", expand=True)
df["reclat"]  = pd.to_numeric(coords[0], errors="coerce")
df["reclong"] = pd.to_numeric(coords[1], errors="coerce")

# 1.6 Convert Year → datetime
df["year"] = pd.to_datetime(df["Year"], errors="coerce")

# 1.7 Check numeric stats
df[["mass_kg", "reclat", "reclong"]].describe()
# Descriptive statistics for key numeric columns
df[['Mass', 'year', 'reclat', 'reclong']].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45716 entries, 0 to 45715
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Name            45716 non-null  object
 1   ID              45716 non-null  int64 
 2   NameType        45716 non-null  object
 3   Classification  45716 non-null  object
 4   Mass            45716 non-null  object
 5   Fall            45716 non-null  object
 6   Year            45716 non-null  object
 7   Coordinates     45716 non-null  object
dtypes: int64(1), object(7)
memory usage: 2.8+ MB


C:\Users\jaush\AppData\Local\Temp\ipykernel_2596\2501631496.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["year"] = pd.to_datetime(df["Year"], errors="coerce")


,year,reclat,reclong
count,0,0.0,0.0
mean,NaT,NaN,NaN
min,NaT,NaN,NaN
25%,NaT,NaN,NaN
50%,NaT,NaN,NaN
75%,NaT,NaN,NaN
max,NaT,NaN,NaN
std,NaN,NaN,NaN


In [9]:
# 2.1 Drop rows missing critical data
df = df.dropna(subset=["mass_kg", "reclat", "reclong", "year", "Fall"])

# 2.2 Filter out non-positive or zero mass
df = df[df["mass_kg"] > 0]

# 2.3 Create a “decade” column for time-based grouping
df["decade"] = (df["year"].dt.year // 10) * 10

In [11]:
# Convert year column to datetime
df['year'] = pd.to_datetime(df['year'], errors='coerce')

# Remove any remaining invalid years or non‐positive masses
df = df.dropna(subset=['year'])
df = df[df['mass_kg'] > 0]

In [13]:
# Mass in kilograms
df['mass_kg'] = df['Mass'] / 1000

# Extract decade for time‐based grouping
df['decade'] = (df['year'].dt.year // 10) * 10

In [14]:
# 10 heaviest meteorites (by mass_kg)
heaviest10 = df.nlargest(10, 'mass_kg')
heaviest10[['name', 'mass_kg', 'year', 'reclat', 'reclong']]

TypeError: Column 'mass_kg' has dtype object, cannot use method 'nlargest' with this dtype

In [ ]:
# Count of  Fell vs. Found
fall_counts = df['Fall'].value_counts()

# Average mass_kg for top 5 most common classifications
top_classes = df['  '].value_counts().head(5).index
avg_mass_by_class = (
    df[df['recclass'].isin(top_classes)]
      .groupby('recclass')['mass_kg']
      .mean()
      .sort_values(ascending=False)
)

KeyError: 'recclass'